<a href="https://colab.research.google.com/github/sr606/Automated-3nf-data-modeling/blob/main/mermaid12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
Final Project Structure
etl_lineage_system/

│
├── main.py
├── .env
├── requirements.txt
│
├── agent_template/
│   └── lineage_engine.py
│
├── lineage_mcp/
│
│   ├── routers/
│   │   └── router.py
│   │
│   ├── tools/
│   │   ├── helpers.py
│   │   └── diagram_generator.py
│   │
│   └── data/
│       ├── upload/
│       └── feature/
1️⃣ requirements.txt
fastapi
uvicorn
python-dotenv

langchain
langchain-openai
langgraph

requests
networkx
tiktoken
2️⃣ .env
AZURE_OPENAI_API_KEY=
AZURE_OPENAI_ENDPOINT=
AZURE_OPENAI_API_VERSION=2024-02-15-preview
AZURE_OPENAI_CHAT_DEPLOYMENT_NAME=gpt-4o
3️⃣ main.py

Runs the FastAPI server.

from fastapi import FastAPI
import uvicorn

from lineage_mcp.routers.router import router as lineage_router

app = FastAPI(title="ETL Lineage System")

app.include_router(lineage_router, prefix="/lineage")


if __name__ == "__main__":

    uvicorn.run(
        "main:app",
        host="0.0.0.0",
        port=8001,
        reload=True
    )

Run server:

python main.py
4️⃣ agent_template/lineage_engine.py

Single agent engine containing:

LLM binding

lineage workflow

tool calling

import os
from dotenv import load_dotenv

from langchain_openai import AzureChatOpenAI

load_dotenv()


class LineageEngine:

    def __init__(self):

        self.llm = AzureChatOpenAI(
            azure_deployment=os.environ["AZURE_OPENAI_CHAT_DEPLOYMENT_NAME"],
            openai_api_version=os.environ["AZURE_OPENAI_API_VERSION"],
            azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
            api_key=os.environ["AZURE_OPENAI_API_KEY"],
            temperature=0.0,
        )

    async def generate_lineage(self, pseudocode):

        prompt = f"""
You are an ETL lineage engineer.

Extract nodes and edges from the following pseudocode.

Return JSON:

nodes: []
edges: []

Pseudocode:
{pseudocode}
"""

        response = await self.llm.ainvoke(prompt)

        return response.content
5️⃣ lineage_mcp/tools/helpers.py

Handles file operations + parsing.

import os
import re

BASE_DIR = "lineage_mcp/data/upload"


def read_file(file_name):

    path = os.path.join(BASE_DIR, file_name)

    with open(path) as f:
        return f.read()


def chunk_text(text, size=2000):

    chunks = []

    for i in range(0, len(text), size):

        chunks.append(text[i:i + size])

    return chunks


def parse_chunks(chunks):

    nodes = set()
    edges = []

    stage_pattern = r"Stage:\s*(\w+)"
    output_pattern = r"Output:\s*(\w+)"

    for chunk in chunks:

        stages = re.findall(stage_pattern, chunk)
        outputs = re.findall(output_pattern, chunk)

        for s in stages:
            nodes.add(s)

        for o in outputs:
            nodes.add(o)

        for i in range(min(len(stages), len(outputs))):

            edges.append((stages[i], outputs[i]))

    return {
        "nodes": list(nodes),
        "edges": edges
    }


def generate_layout(graph):

    nodes = []
    x = 100
    y = 100

    for node in graph["nodes"]:

        nodes.append({
            "id": node,
            "x": x,
            "y": y
        })

        y += 120

    return {
        "nodes": nodes,
        "edges": graph["edges"]
    }
6️⃣ lineage_mcp/tools/diagram_generator.py

Creates draw.io diagram.

import os
import uuid

OUTPUT_DIR = "lineage_mcp/data/feature"


def create_drawio_diagram(nodes, edges, file_name):

    node_ids = {}
    xml = []

    xml.append("<mxfile><diagram><mxGraphModel><root>")
    xml.append('<mxCell id="0"/><mxCell id="1" parent="0"/>')

    for node in nodes:

        nid = str(uuid.uuid4())
        node_ids[node["id"]] = nid

        xml.append(
            f'<mxCell id="{nid}" value="{node["id"]}" vertex="1" parent="1">'
            f'<mxGeometry x="{node["x"]}" y="{node["y"]}" width="140" height="60" as="geometry"/>'
            '</mxCell>'
        )

    for src, dst in edges:

        xml.append(
            f'<mxCell edge="1" parent="1" source="{node_ids[src]}" target="{node_ids[dst]}">'
            '<mxGeometry relative="1" as="geometry"/>'
            '</mxCell>'
        )

    xml.append("</root></mxGraphModel></diagram></mxfile>")

    path = os.path.join(OUTPUT_DIR, file_name)

    with open(path, "w") as f:
        f.write("\n".join(xml))

    return path
7️⃣ lineage_mcp/routers/router.py

FastAPI routes.

from fastapi import APIRouter

from agent_template.lineage_engine import LineageEngine
from lineage_mcp.tools.helpers import (
    read_file,
    chunk_text,
    parse_chunks,
    generate_layout
)

from lineage_mcp.tools.diagram_generator import create_drawio_diagram

router = APIRouter()

engine = LineageEngine()


@router.post("/generate")
async def generate_lineage(file_name: str):

    text = read_file(file_name)

    chunks = chunk_text(text)

    graph = parse_chunks(chunks)

    layout = generate_layout(graph)

    diagram = create_drawio_diagram(
        layout["nodes"],
        layout["edges"],
        "lineage.drawio"
    )

    return {
        "diagram": diagram
    }